In [0]:
# Spark optimizations for large write operations
# On serverless: AQE, Photon, shuffle optimization, and optimizeWrite are all
# enabled by default. Only set Delta-specific configs that are user-configurable.
for key, val in {
    "spark.sql.shuffle.partitions": "auto",
    "spark.databricks.delta.optimizeWrite.enabled": "true",
}.items():
    try:
        spark.conf.set(key, val)
        print(f"  Set {key} = {val}")
    except Exception:
        print(f"  Skip {key} (already managed by serverless)")

In [0]:
%sql
SELECT session_start_hour, COUNT(*)
FROM dev.detection.viewing_commercials_golden
WHERE session_start_hour >= '2026-09-01 00:00:00'
  AND session_start_hour < '2026-09-04 00:00:00'
GROUP BY 1
ORDER BY 1 DESC

In [0]:
from datetime import datetime, timedelta
import time

# ── Configuration ──────────────────────────────────────────
START_DATE  = "2026-06-15"  # 3 months ago
END_DATE    = "2026-09-01"

SOURCE_TABLE = "prod.detection.viewing_commercials_golden"
TARGET_TABLE = "dev.detection.viewing_commercials_golden"

# Align column order: prod and dev have different column positions
target_columns = spark.table(TARGET_TABLE).columns

# Build date list
dates_to_backfill = []
cur = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")
while cur < end:
    dates_to_backfill.append(cur.strftime("%Y-%m-%d"))
    cur += timedelta(days=1)


def backfill_day(date_str):
    """Backfill one day using SQL INSERT INTO REPLACE WHERE (faster than DataFrame writer)."""
    day_start = f"{date_str} 00:00:00"
    next_day  = (datetime.strptime(date_str, "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-%d")
    day_end   = f"{next_day} 00:00:00"
    condition = f"session_start_hour >= '{day_start}' AND session_start_hour < '{day_end}'"
    cols = ", ".join(target_columns)

    t0 = time.time()
    spark.sql(f"""
        INSERT INTO {TARGET_TABLE}
        REPLACE WHERE {condition}
        SELECT {cols}
        FROM {SOURCE_TABLE}
        WHERE {condition}
    """)

    return {"date": date_str, "minutes": round((time.time() - t0) / 60, 1)}


# ── Summary ────────────────────────────────────────────────
print(f"Backfill Configuration")
print(f"{'─' * 55}")
print(f"  Source:     {SOURCE_TABLE}")
print(f"  Target:     {TARGET_TABLE}")
print(f"  Range:      {START_DATE} → {END_DATE} ({len(dates_to_backfill)} days)")
print(f"  Est. rows:  ~{len(dates_to_backfill) * 9}B")
print(f"  Strategy:   SQL INSERT INTO REPLACE WHERE (per day, idempotent)")

In [0]:
# ── Execute sequential daily backfill ─────────────────────
# NOTE: This will process ~86 days × ~9B rows/day sequentially. Expect several hours.
results, failed = [], []
total = len(dates_to_backfill)

print(f"Starting backfill: {total} days (sequential)\n")
t_start = time.time()

for i, d in enumerate(dates_to_backfill, 1):
    pct = i / total * 100
    try:
        r = backfill_day(d)
        results.append(r)
        elapsed_so_far = (time.time() - t_start) / 3600
        est_total = elapsed_so_far / i * total
        print(f"  [{i:3d}/{total}] {pct:5.1f}%  ✓ {r['date']}  {r['minutes']:.1f} min  (ETA: {est_total:.1f}h total)")
    except Exception as e:
        failed.append({"date": d, "error": str(e)})
        print(f"  [{i:3d}/{total}] {pct:5.1f}%  ✗ {d}  FAILED: {e}")

elapsed_h = (time.time() - t_start) / 3600
print(f"\n{'═' * 55}")
print(f"Done  -  {len(results)} OK, {len(failed)} failed  -  {elapsed_h:.1f} hours total")
if failed:
    print(f"\nFailed dates (re-run with dates_to_backfill = [...]):")
    for f in sorted(failed, key=lambda x: x['date']):
        print(f"  {f['date']}: {f['error']}")

In [0]:
# ── Retry any failed dates ─────────────────────────────────
# Uncomment and edit the list below to retry specific dates,
# or run as-is to retry all dates that failed in the previous cell.

# dates_to_backfill = ["2026-07-15", "2026-08-03"]  # ← edit to retry specific dates

if failed:
    retry_dates = [f["date"] for f in failed]
    print(f"Retrying {len(retry_dates)} failed dates...\n")
    still_failed = []

    for i, d in enumerate(retry_dates, 1):
        try:
            r = backfill_day(d)
            print(f"  [{i}/{len(retry_dates)}] ✓ {r['date']}  {r['minutes']:.1f} min")
        except Exception as e:
            still_failed.append({"date": d, "error": str(e)})
            print(f"  [{i}/{len(retry_dates)}] ✗ {d}  STILL FAILED: {e}")

    if still_failed:
        print(f"\nPersistent failures: {[f['date'] for f in still_failed]}")
    else:
        print("\nAll retries succeeded!")
else:
    print("No failed dates to retry.")

In [0]:
from datetime import datetime, timedelta
import time

SOURCE_TABLE = "prod.detection.viewing_commercials_golden"
TARGET_TABLE = "dev.detection.viewing_commercials_golden"
target_columns = spark.table(TARGET_TABLE).columns

# ── Gap ranges to fill (inclusive start, exclusive end) ────
gap_ranges = [
    ("2026-09-01 00:00:00", "2026-09-02 20:00:00"),  # Sep 1 00:00 → Sep 2 20:00
    ("2026-09-09 19:00:00", "2026-09-10 13:00:00"),  # Sep 9 19:00 → Sep 10 12:00
]

# Build list of individual hours to backfill
hours_to_backfill = []
for range_start, range_end in gap_ranges:
    cur = datetime.strptime(range_start, "%Y-%m-%d %H:%M:%S")
    end = datetime.strptime(range_end, "%Y-%m-%d %H:%M:%S")
    while cur < end:
        hours_to_backfill.append(cur.strftime("%Y-%m-%d %H:%M:%S"))
        cur += timedelta(hours=1)


def backfill_hour(hour_str):
    """Backfill a single hour using SQL INSERT INTO REPLACE WHERE."""
    condition = f"session_start_hour = '{hour_str}'"
    cols = ", ".join(target_columns)

    t0 = time.time()
    spark.sql(f"""
        INSERT INTO {TARGET_TABLE}
        REPLACE WHERE {condition}
        SELECT {cols}
        FROM {SOURCE_TABLE}
        WHERE {condition}
    """)
    return {"hour": hour_str, "minutes": round((time.time() - t0) / 60, 1)}

# ── Summary ────────────────────────────────────────────────
print(f"Backfill Configuration")
print(f"{'─' * 55}")
print(f"  Source:     {SOURCE_TABLE}")
print(f"  Target:     {TARGET_TABLE}")
print(f"  Hours to backfill:      {hours_to_backfill} ({len(hours_to_backfill)} hours)")
print(f"  Strategy:   SQL INSERT INTO REPLACE WHERE (per hour, idempotent)")

In [0]:
# ── Execute ────────────────────────────────────────────────
results, failed = [], []
total = len(hours_to_backfill)
print(f"Backfilling {total} missing hours\n")
t_start = time.time()

for i, h in enumerate(hours_to_backfill, 1):
    pct = i / total * 100
    try:
        r = backfill_hour(h)
        results.append(r)
        elapsed_so_far = (time.time() - t_start) / 3600
        est_total = elapsed_so_far / i * total
        print(f"  [{i:3d}/{total}] {pct:5.1f}%  \u2713 {r['hour']}  {r['minutes']:.1f} min  (ETA: {est_total:.1f}h total)")
    except Exception as e:
        failed.append({"hour": h, "error": str(e)})
        print(f"  [{i:3d}/{total}] {pct:5.1f}%  \u2717 {h}  FAILED: {e}")

elapsed_h = (time.time() - t_start) / 3600
print(f"\n{'\u2550' * 55}")
print(f"Done  \u2013  {len(results)} OK, {len(failed)} failed  \u2013  {elapsed_h:.1f} hours total")
if failed:
    print(f"\nFailed hours:")
    for f in sorted(failed, key=lambda x: x['hour']):
        print(f"  {f['hour']}: {f['error']}")

In [0]:
# ── Spot-check: compare row counts for first and last 7 days ──
from pyspark.sql.functions import count, substring, col

check_ranges = [
    (START_DATE, (datetime.strptime(START_DATE, "%Y-%m-%d") + timedelta(days=7)).strftime("%Y-%m-%d")),
    ((datetime.strptime(END_DATE, "%Y-%m-%d") - timedelta(days=7)).strftime("%Y-%m-%d"), END_DATE),
]

for rng_start, rng_end in check_ranges:
    cond = f"session_start_hour >= '{rng_start} 00:00:00' AND session_start_hour < '{rng_end} 00:00:00'"

    prod_df = (spark.table(SOURCE_TABLE).filter(cond)
               .withColumn("day", substring("session_start_hour", 1, 10))
               .groupBy("day").agg(count("*").alias("prod_rows")))

    dev_df  = (spark.table(TARGET_TABLE).filter(cond)
               .withColumn("day", substring("session_start_hour", 1, 10))
               .groupBy("day").agg(count("*").alias("dev_rows")))

    comp = (prod_df.join(dev_df, "day", "full")
            .withColumn("match", col("prod_rows") == col("dev_rows"))
            .orderBy("day"))

    print(f"\nValidation: {rng_start} → {rng_end}")
    display(comp)